## Regional SAT anomalies calculation then calculate the trend

In [1]:
# In[1]:
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
# %%
# define function
import src.SAT_function_Obs_Fingerprint as data_process
import src.Data_Preprocess as preprocess

In [2]:
# import src.slurm_cluster as scluster
# client, scluster = scluster.init_dask_slurm_cluster(scale=2,cores=10, memory="200GB")

In [3]:
def func_mk(x):
    """
    Mann-Kendall test for trend
    """
    results = data_process.mk_test(x)
    slope = results[0]
    p_val = results[1]
    return slope, p_val

In [4]:
dir_in = '/work/mh0033/m301036/OBS_LPS_revision/docs/data/FIG3/MMLE/SMILE_forced_2013/'
MMLE_TREND = xr.open_dataset(dir_in + 'MMEM_ENSmean_forced_MK_trend_1950-2013_sliding.nc')

In [5]:
MMLE_TREND

<xarray.Dataset>
Dimensions:  (lon: 180, lat: 90, period: 55)
Coordinates:
  * lon      (lon) float64 0.0 2.0 4.0 6.0 8.0 ... 350.0 352.0 354.0 356.0 358.0
  * lat      (lat) float64 -89.0 -87.0 -85.0 -83.0 -81.0 ... 83.0 85.0 87.0 89.0
  * period   (period) object '1950-2013' '1951-2013' ... '2003-2013' '2004-2013'
Data variables:
    trend    (period, lat, lon) float64 ...
    p_value  (period, lat, lon) float64 ...

### Calculate the trend end year fix to 2022, start with 73 year length and decrease length of trend every one year, the minimum trend length is 10yr 

In [6]:
temp_data = MMLE_TREND.trend

### Regional anomalies calculation

In [7]:
def plot_trend(temp_data, lats, lons, levels=None, extend=None, cmap=None, 
                                 title="", ax=None, show_xticks=False, show_yticks=False):
    """
    Plot the trend spatial pattern using Robinson projection with significance overlaid.

    Parameters:
    - temp_data: 2D numpy array with the trend values.
    - lats, lons: 1D arrays of latitudes and longitudes.
    - p_values: 2D array with p-values for each grid point.
    - GMST_p_values: 2D array with GMST p-values for each grid point.
    - title: Title for the plot.
    - ax: Existing axis to plot on. If None, a new axis will be created.
    - show_xticks, show_yticks: Boolean flags to show x and y axis ticks.
    
    Returns:
    - contour_obj: The contour object from the plot.
    """
    # Plotting
    contour_obj = ax.contourf(lons, lats, temp_data, levels=levels, extend=extend, cmap=cmap, transform=ccrs.PlateCarree())

    ax.coastlines(resolution='110m')
    gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False,
                      color='gray', alpha=0.35, linestyle='--')

    # Disable labels on the top and right of the plot
    gl.top_labels = False
    gl.right_labels = False

    # Enable labels on the bottom and left of the plot
    gl.bottom_labels = show_xticks
    gl.left_labels = show_yticks
    gl.xformatter = cticker.LongitudeFormatter()
    gl.yformatter = cticker.LatitudeFormatter()
    gl.xlabel_style = {'size': 16}
    gl.ylabel_style = {'size': 16}
    
    if show_xticks:
        gl.bottom_labels = True
    if show_yticks:
        gl.left_labels = True
    
    # ax.set_title(title, loc='center', fontsize=18, pad=5.0)

    return contour_obj
# %%
plt.rcParams['figure.figsize'] = (8, 10)
plt.rcParams['font.size'] = 16
# plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.labelsize'] = 16
plt.rcParams['ytick.direction'] = 'out'
plt.rcParams['ytick.minor.visible'] = True
plt.rcParams['ytick.major.right'] = True
plt.rcParams['ytick.right'] = True
plt.rcParams['xtick.bottom'] = True
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['savefig.bbox'] = 'tight'
plt.rcParams['savefig.pad_inches'] = 0.1
plt.rcParams['savefig.transparent'] = True

import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.ticker as mticker
import cartopy.feature as cfeature
import cartopy.mpl.ticker as cticker
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import matplotlib.gridspec as gridspec
import matplotlib as mpl
import seaborn as sns
from matplotlib.colors import ListedColormap
from matplotlib.colors import BoundaryNorm, ListedColormap
import cartopy.util as cutil
import seaborn as sns
import matplotlib.colors as mcolors
import palettable

In [8]:
lat = temp_data.lat
lon = temp_data.lon

In [9]:
# Southern Ocean Pacific sector region
lat1 = -70
lat2 = -55
lon1 = 180
lon2 = 260

In [10]:
temp_data_SOP,lons_SOP, lats_SOP = data_process.selreg(
        temp_data, lat, lon, lat1=lat1, lat2=lat2, lon1=lon1, 
        lon2=lon2)

In [11]:
# calculate the SOP region SAT anomalies
temp_da_SOP_mean = data_process.calc_weighted_mean(temp_data_SOP)

In [12]:
temp_da_SOP_mean

<xarray.DataArray 'trend' (period: 55)>
array([0.07165437, 0.07387506, 0.07578515, 0.07773462, 0.0796336 ,
       0.08093489, 0.08250201, 0.08354892, 0.08588998, 0.08778799,
       0.08927827, 0.09077583, 0.09300747, 0.09503214, 0.09723582,
       0.09691569, 0.09880156, 0.1001258 , 0.10059167, 0.10245869,
       0.10312808, 0.10217179, 0.10300702, 0.10544261, 0.10863651,
       0.11186575, 0.11440144, 0.11466556, 0.1136637 , 0.11498991,
       0.11693881, 0.12000444, 0.12200811, 0.12446974, 0.12553045,
       0.12478399, 0.12447088, 0.12719378, 0.13425018, 0.14232321,
       0.15724861, 0.17040257, 0.18505355, 0.19709337, 0.19917832,
       0.18567758, 0.17107896, 0.1588542 , 0.14879071, 0.14241581,
       0.12617822, 0.12610285, 0.14452011, 0.16145187, 0.19160259])
Coordinates:
  * period   (period) object '1950-2013' '1951-2013' ... '2003-2013' '2004-2013'

In [13]:
import os

dir_out = '/work/mh0033/m301036/OBS_LPS_revision/docs/data/FIG5/end_year_2013/MMLE/'
os.makedirs(dir_out, exist_ok=True)

temp_da_SOP_mean.to_dataset(name='trend').to_netcdf(dir_out + 'MMLE_ENSforced_SOP_trend_1950-2013_sliding.nc')

In [14]:
temp_da_SOP_mean

<xarray.DataArray 'trend' (period: 55)>
array([0.07165437, 0.07387506, 0.07578515, 0.07773462, 0.0796336 ,
       0.08093489, 0.08250201, 0.08354892, 0.08588998, 0.08778799,
       0.08927827, 0.09077583, 0.09300747, 0.09503214, 0.09723582,
       0.09691569, 0.09880156, 0.1001258 , 0.10059167, 0.10245869,
       0.10312808, 0.10217179, 0.10300702, 0.10544261, 0.10863651,
       0.11186575, 0.11440144, 0.11466556, 0.1136637 , 0.11498991,
       0.11693881, 0.12000444, 0.12200811, 0.12446974, 0.12553045,
       0.12478399, 0.12447088, 0.12719378, 0.13425018, 0.14232321,
       0.15724861, 0.17040257, 0.18505355, 0.19709337, 0.19917832,
       0.18567758, 0.17107896, 0.1588542 , 0.14879071, 0.14241581,
       0.12617822, 0.12610285, 0.14452011, 0.16145187, 0.19160259])
Coordinates:
  * period   (period) object '1950-2013' '1951-2013' ... '2003-2013' '2004-2013'

In [ ]:
da_plot = temp_data.sel(period="1950-2022")

fig, ax = plt.subplots(subplot_kw={'projection': ccrs.Robinson()})

levels = np.arange(-0.5, 0.55, 0.05)
contour_obj =  plot_trend(da_plot, da_plot.lat, da_plot.lon, levels=levels, extend='both', cmap='RdBu_r',
                                    title="SEP region SAT anomaly 2020", ax=ax, show_xticks=True, show_yticks=True)

# colorbar
cbar = plt.colorbar(contour_obj, ax=ax, orientation='horizontal', pad=0.05, aspect=50)
cbar.set_label('Temperature anomaly (°C)')
cbar.ax.tick_params(labelsize=14)

plt.show()